# Customer Segmentation: Understanding Who Your Customers Are Before Predicting What They Will Do

Churn prediction tells you which customers are likely to leave. Segmentation tells you who they are. This project applies unsupervised learning to the same IBM Telco dataset used in telco-churn-predictor — not to predict churn, but to discover the natural customer archetypes that exist in the data. A retention strategy that treats all at-risk customers identically is less effective than one tailored to each segment's specific profile and motivations.

In [ ]:
import json
import pickle
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.offline as pyo
import seaborn as sns
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
%matplotlib inline

MODEL_DIR = Path('models')
PLOTS_DIR = Path('plots')
RANDOM_STATE = 42
N_CLUSTERS = 5

MODEL_DIR.mkdir(exist_ok=True)
PLOTS_DIR.mkdir(exist_ok=True)

print('Setup complete.')

## Dataset Overview

This is the same dataset used in my churn prediction project. Here I use it differently — Churn is excluded from the clustering features and used only as a post-hoc validation label to check whether the segments we discover correspond to meaningful differences in churn behaviour.

The IBM Telco Customer Churn dataset contains 7,043 customers across 21 columns covering demographics, service subscriptions, account details, and whether the customer churned. The task of telco-churn-predictor was to predict that final column. The task here is to ignore it entirely during modelling and discover what natural groupings emerge from how customers actually use and pay for the service.

In [ ]:
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)
print(f'Raw shape: {df.shape}')
print(f'\nColumns: {df.columns.tolist()}')

# Apply same cleaning as telco-churn-predictor
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
n_dropped = df['TotalCharges'].isna().sum()
df.dropna(subset=['TotalCharges'], inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df.drop(columns=['customerID'], inplace=True)

print(f'\nDropped {n_dropped} rows with missing TotalCharges.')
print(f'Cleaned shape: {df.shape}')
df.head()

## Exploratory Data Analysis

Before clustering, I want to understand the distributions of the key numeric features and the correlations between them. These EDA charts inform feature selection and help identify any preprocessing needs (skewed distributions, outliers).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['tenure', 'MonthlyCharges', 'TotalCharges']):
    ax.hist(df[col], bins=30, color='#2E75B6', edgecolor='white', alpha=0.85)
    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
fig.suptitle('Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '01_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix — Numeric Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '02_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Feature Selection for Clustering

Features were selected for their business interpretability and their relevance to customer behaviour — not just statistical variance. Gender, Partner, and Dependents were excluded because they describe who the customer is demographically, not how they use the service. Service usage patterns (internet type, tech support, security) and commercial relationship (contract, tenure, charges) are more actionable segmentation dimensions for a retention team.

The `Churn` column is excluded from clustering features entirely — it is used only after clustering as a post-hoc validation label to check whether the segments correspond to meaningful differences in churn risk. Including it would make the segmentation circular: we would be discovering "churners vs non-churners" rather than natural behavioural archetypes.

**Features included:**
- `tenure` — captures customer lifecycle stage
- `MonthlyCharges` — captures spending level
- `TotalCharges` — captures lifetime value
- `SeniorCitizen` — demographic signal
- `Contract` — strongest churn predictor from telco project; captures commitment level
- `InternetService` — service tier signal
- `TechSupport` — engagement signal
- `OnlineSecurity` — engagement signal
- `PaymentMethod` — payment behaviour signal
- `PaperlessBilling` — digital engagement signal

In [ ]:
churn_labels = df['Churn'].copy()

binary_map = {'Yes': 1, 'No': 0, 'No phone service': 0, 'No internet service': 0}
binary_cols = ['Partner', 'Dependents', 'PhoneService', 'PaperlessBilling',
               'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
               'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']
for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].map(binary_map).fillna(df[col])

cluster_features = [
    'tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen',
    'Contract', 'InternetService', 'TechSupport', 'OnlineSecurity',
    'PaymentMethod', 'PaperlessBilling'
]

df_cluster = df[cluster_features].copy()
df_cluster = pd.get_dummies(df_cluster,
                            columns=['Contract', 'InternetService', 'PaymentMethod'],
                            drop_first=True)

feature_names = df_cluster.columns.tolist()
print(f'Clustering features ({len(feature_names)} total):')
for f in feature_names:
    print(f'  {f}')

# Clustering is distance-based — unscaled features dominate distance calculations.
# StandardScaler ensures every feature contributes proportionally.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster)

pickle.dump(scaler, open(MODEL_DIR / 'scaler.pkl', 'wb'))
pickle.dump(feature_names, open(MODEL_DIR / 'feature_names.pkl', 'wb'))
print(f'\nScaled feature matrix shape: {X_scaled.shape}')

## Finding Optimal k

Neither the elbow method nor the silhouette score gives a single definitive answer — they are guides. The elbow method finds where adding more clusters gives diminishing returns in terms of within-cluster variance. The silhouette score measures cluster cohesion and separation. When both methods agree, we have strong evidence for k. When they disagree, domain knowledge and interpretability should guide the final decision — a segmentation with 5 interpretable archetypes is more useful to a business client than one with 7 mathematically optimal but indistinguishable clusters.

In [ ]:
k_range = range(2, 11)
inertias = []
silhouette_scores_list = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouette_scores_list.append(
        silhouette_score(X_scaled, labels, sample_size=2000, random_state=RANDOM_STATE)
    )

best_k_silhouette = list(k_range)[np.argmax(silhouette_scores_list)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(list(k_range), inertias, 'o-', color='#2E75B6', linewidth=2, markersize=7)
ax1.axvline(x=N_CLUSTERS, color='#E74C3C', linestyle='--', linewidth=1.5,
            label=f'k={N_CLUSTERS} (selected)')
ax1.set_xlabel('Number of Clusters (k)')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(list(k_range), silhouette_scores_list, 's-', color='#2ECC71', linewidth=2, markersize=7)
ax2.axvline(x=best_k_silhouette, color='#E74C3C', linestyle='--', linewidth=1.5,
            label=f'Best k={best_k_silhouette}')
ax2.set_xlabel('Number of Clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Finding Optimal k', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '03_elbow_method.png', dpi=150, bbox_inches='tight')
plt.savefig(PLOTS_DIR / '04_silhouette_scores.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Elbow suggests: k={N_CLUSTERS} (selected)')
print(f'Silhouette suggests: k={best_k_silhouette}')

## KMeans vs Hierarchical Clustering

**K-Means** assigns points to the nearest centroid and iterates until stable. Fast and scalable, requires k upfront, and is sensitive to initialisation — hence `n_init=10`, which runs the algorithm 10 times and keeps the best result by inertia.

**Hierarchical clustering** builds a tree of merges bottom-up, starting with every point as its own cluster. Slower but does not require k upfront and produces a dendrogram that visualises the full merge history. Ward linkage minimises within-cluster variance at each merge — equivalent objective to K-Means but a different algorithm.

For 7,043 customers, K-Means is the practical choice for deployment. Hierarchical clustering is used here as a validation tool — if both algorithms agree on the cluster structure (high Adjusted Rand Score), we have more confidence the segments are real features of the data rather than artefacts of the algorithm.

In [ ]:
# KMeans
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)
df['KMeans_Cluster'] = kmeans_labels
pickle.dump(kmeans, open(MODEL_DIR / 'kmeans_model.pkl', 'wb'))
print('KMeans cluster distribution:')
for c in range(N_CLUSTERS):
    n = (kmeans_labels == c).sum()
    print(f'  Cluster {c}: {n} ({n/len(kmeans_labels)*100:.1f}%)')

# Hierarchical
agg = AgglomerativeClustering(n_clusters=N_CLUSTERS, linkage='ward')
hier_labels = agg.fit_predict(X_scaled)
df['Hierarchical_Cluster'] = hier_labels

ari = adjusted_rand_score(kmeans_labels, hier_labels)
pct = (kmeans_labels == hier_labels).mean() * 100
print(f'\nKMeans and Hierarchical agree on {pct:.1f}% of assignments')
print(f'Adjusted Rand Score: {ari:.3f}')

# Dendrogram
sample_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_scaled), 200, replace=False)
Z = linkage(X_scaled[sample_idx], method='ward')
cut_height = sorted(Z[:, 2], reverse=True)[N_CLUSTERS - 1]

fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(Z, ax=ax, truncate_mode='lastp', p=20, leaf_font_size=9, color_threshold=0)
ax.axhline(y=cut_height, color='#E74C3C', linestyle='--', linewidth=1.5,
           label=f'Cut for k={N_CLUSTERS}')
ax.set_title('Hierarchical Clustering Dendrogram (200-row sample, Ward linkage)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Sample index (or cluster size)')
ax.set_ylabel('Ward linkage distance')
ax.legend()
plt.tight_layout()
plt.savefig(PLOTS_DIR / '05_dendrogram.png', dpi=150, bbox_inches='tight')
plt.show()

## PCA for Visualisation

PCA does not change the clustering — it is applied after clustering purely for visualisation. We cannot plot a 20-dimensional space, but we can project it to 2 or 3 dimensions that capture the most variance and check whether the clusters are visually separable. Well-separated clusters in PCA space give us confidence the segments are distinct.

I fit the clusters in the full feature space and use PCA only to project to 2D/3D. Applying PCA before clustering would reduce interpretability of the original features: the cluster centroids would be defined in principal component space, not in business terms like "average tenure 48 months, two-year contract."

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled)
pickle.dump(pca_full, open(MODEL_DIR / 'pca_model.pkl', 'wb'))

evr = pca_full.explained_variance_ratio_
cumulative = np.cumsum(evr)

# PCA explained variance chart
fig, ax = plt.subplots(figsize=(10, 5))
x_pos = range(1, 11)
ax.bar(x_pos, evr[:10] * 100, color='#2E75B6', alpha=0.8, label='Individual')
ax2 = ax.twinx()
ax2.plot(x_pos, cumulative[:10] * 100, 'o-', color='#E74C3C', linewidth=2, label='Cumulative')
ax2.axhline(y=80, color='#E74C3C', linestyle='--', linewidth=1, alpha=0.6)
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance (%)')
ax2.set_ylabel('Cumulative Explained Variance (%)')
ax.set_title('PCA Explained Variance', fontsize=13, fontweight='bold')
ax.set_xticks(list(x_pos))
lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, loc='center right')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '06_pca_explained_variance.png', dpi=150, bbox_inches='tight')
plt.show()

print('Explained variance (first 5 components):')
for i, v in enumerate(evr[:5]):
    print(f'  PC{i+1}: {v*100:.1f}%  (cumulative: {cumulative[i]*100:.1f}%)')

# 2D scatter
cluster_colours = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12', '#9B59B6']
X_2d = pca_full.transform(X_scaled)[:, :2]
pc1_var, pc2_var = evr[0]*100, evr[1]*100

fig, ax = plt.subplots(figsize=(10, 7))
for c in range(N_CLUSTERS):
    mask = kmeans_labels == c
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               c=cluster_colours[c], alpha=0.4, s=15, label=f'Cluster {c}')
centroids_2d = pca_full.transform(kmeans.cluster_centers_)[:, :2]
ax.scatter(centroids_2d[:, 0], centroids_2d[:, 1],
           marker='X', s=200, c='black', zorder=5, label='Centroids')
ax.set_xlabel(f'Principal Component 1 ({pc1_var:.1f}% variance)')
ax.set_ylabel(f'Principal Component 2 ({pc2_var:.1f}% variance)')
ax.set_title('Customer Segments — 2D PCA Projection', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '07_clusters_2d.png', dpi=150, bbox_inches='tight')
plt.show()

# Interactive 3D (Plotly)
X_3d = pca_full.transform(X_scaled)[:, :3]
pc3_var = evr[2]*100
plot_df = pd.DataFrame({
    'PC1': X_3d[:, 0], 'PC2': X_3d[:, 1], 'PC3': X_3d[:, 2],
    'Cluster': [f'Cluster {c}' for c in kmeans_labels]
})
fig_3d = px.scatter_3d(
    plot_df, x='PC1', y='PC2', z='PC3', color='Cluster',
    color_discrete_sequence=cluster_colours,
    title='Customer Segments — 3D PCA Projection (interactive)',
    labels={'PC1': f'PC1 ({pc1_var:.1f}%)', 'PC2': f'PC2 ({pc2_var:.1f}%)',
            'PC3': f'PC3 ({pc3_var:.1f}%)'}, opacity=0.6
)
fig_3d.update_traces(marker=dict(size=3))
fig_3d.write_html(str(PLOTS_DIR / '08_clusters_3d.html'))
pyo.iplot(fig_3d)

## Cluster Profiles

The cluster numbers (0, 1, 2...) are arbitrary labels assigned by the algorithm. What matters is the profile of each cluster — its typical tenure, spend, contract type, and crucially, its churn rate. High churn rate in a cluster does not mean we should target them for churn prevention — it means we need to understand why they churn and whether retention is even the right strategy for that segment.

A month-to-month Fiber optic customer on electronic check with no add-on services who churns after 3 months may be a price-shopper who responds to promotional offers. A two-year contract customer with full services who churns after year two may be leaving because of a life event. The same churn rate, completely different retention levers.

In [ ]:
overall_churn_rate = churn_labels.mean() * 100
cluster_profiles = {}
profile_rows = []

for c in range(N_CLUSTERS):
    mask = df['KMeans_Cluster'] == c
    sub = df[mask]
    churn_sub = churn_labels[mask]
    size = mask.sum()
    cluster_profiles[str(c)] = {
        'size': int(size),
        'pct_total': round(size / len(df) * 100, 1),
        'mean_tenure': round(sub['tenure'].mean(), 1),
        'mean_monthly_charges': round(sub['MonthlyCharges'].mean(), 2),
        'mean_total_charges': round(sub['TotalCharges'].mean(), 2),
        'churn_rate': round(churn_sub.mean() * 100, 1),
        'most_common_contract': sub['Contract'].mode()[0],
        'most_common_internet': sub['InternetService'].mode()[0],
        'most_common_payment': sub['PaymentMethod'].mode()[0]
    }
    profile_rows.append({
        'Cluster': c,
        'Size': size,
        'Tenure': round(sub['tenure'].mean(), 1),
        'Monthly$': round(sub['MonthlyCharges'].mean(), 2),
        'Total$': round(sub['TotalCharges'].mean(), 2),
        'Churn%': round(churn_sub.mean() * 100, 1),
        'Contract': sub['Contract'].mode()[0],
        'Internet': sub['InternetService'].mode()[0]
    })

profile_df = pd.DataFrame(profile_rows).set_index('Cluster')
print('Cluster Profile Table:')
print(profile_df.to_string())
print(f'\nOverall churn rate: {overall_churn_rate:.1f}%')

with open(MODEL_DIR / 'cluster_profiles.json', 'w') as f:
    json.dump(cluster_profiles, f, indent=2)

df.to_csv(MODEL_DIR / 'clustered_data.csv', index=False)

pca_variance_data = {
    'explained_variance_ratio': evr[:10].tolist(),
    'cumulative_variance': cumulative[:10].tolist()
}
with open(MODEL_DIR / 'pca_variance.json', 'w') as f:
    json.dump(pca_variance_data, f, indent=2)

# Cluster profiles chart (2x3 grid)
metrics = ['tenure', 'MonthlyCharges', 'TotalCharges']
metric_keys = ['mean_tenure', 'mean_monthly_charges', 'mean_total_charges']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, (metric, key) in enumerate(zip(metrics, metric_keys)):
    vals = [cluster_profiles[str(c)][key] for c in range(N_CLUSTERS)]
    axes[i].bar(range(N_CLUSTERS), vals, color=cluster_colours[:N_CLUSTERS], edgecolor='white')
    axes[i].set_title(f'Mean {metric}', fontweight='bold')
    axes[i].set_xlabel('Cluster')
    axes[i].set_xticks(range(N_CLUSTERS))

churn_vals = [cluster_profiles[str(c)]['churn_rate'] for c in range(N_CLUSTERS)]
axes[3].bar(range(N_CLUSTERS), churn_vals, color=cluster_colours[:N_CLUSTERS], edgecolor='white')
axes[3].axhline(overall_churn_rate, color='black', linestyle='--', linewidth=1.2,
                label=f'Overall {overall_churn_rate:.1f}%')
axes[3].set_title('Churn Rate (%)', fontweight='bold')
axes[3].set_xlabel('Cluster')
axes[3].set_xticks(range(N_CLUSTERS))
axes[3].legend()

contract_data = {}
for c in range(N_CLUSTERS):
    mask = df['KMeans_Cluster'] == c
    contract_data[c] = df[mask]['Contract'].value_counts(normalize=True) * 100
contract_df = pd.DataFrame(contract_data).fillna(0).T
contract_df.plot(kind='bar', ax=axes[4], colormap='tab10', edgecolor='white')
axes[4].set_title('Contract Type Distribution (%)', fontweight='bold')
axes[4].set_xlabel('Cluster')
axes[4].set_xticklabels(range(N_CLUSTERS), rotation=0)
axes[4].legend(fontsize=7)

internet_data = {}
for c in range(N_CLUSTERS):
    mask = df['KMeans_Cluster'] == c
    internet_data[c] = df[mask]['InternetService'].value_counts(normalize=True) * 100
internet_df = pd.DataFrame(internet_data).fillna(0).T
internet_df.plot(kind='bar', ax=axes[5], colormap='Set2', edgecolor='white')
axes[5].set_title('Internet Service Distribution (%)', fontweight='bold')
axes[5].set_xlabel('Cluster')
axes[5].set_xticklabels(range(N_CLUSTERS), rotation=0)
axes[5].legend(fontsize=7)

fig.suptitle('Cluster Profiles — Key Metrics by Segment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '09_cluster_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

# Churn by cluster
sorted_clusters = sorted(range(N_CLUSTERS), key=lambda c: churn_vals[c], reverse=True)
sorted_churn = [churn_vals[c] for c in sorted_clusters]
bar_colours = ['#E74C3C' if v > overall_churn_rate else '#2ECC71' for v in sorted_churn]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar([f'Cluster {c}' for c in sorted_clusters], sorted_churn,
              color=bar_colours, edgecolor='white')
ax.axhline(overall_churn_rate, color='#2C3E50', linestyle='--', linewidth=1.5,
           label=f'Overall {overall_churn_rate:.1f}%')
ax.set_xlabel('Customer Segment')
ax.set_ylabel('Churn Rate (%)')
ax.set_title('Churn Rate by Customer Segment', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
for bar, val in zip(bars, sorted_churn):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '10_churn_by_cluster.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nACTION REQUIRED: Based on the cluster profiles above, open')
print(f'segment_labels.py and fill in the TBD values for each cluster.')
print(f'Consider: What is the defining characteristic of this cluster?')
print(f'What is its churn risk relative to the dataset average ({overall_churn_rate:.1f}%)?')
print(f'What retention action would you recommend for this segment?')

## Business Recommendations

The following recommendations are written for a telco client. Specific cluster numbers and statistics are TBD — fill in after running the full training pipeline and labelling the segments.

**1. Protect the stable revenue base.**
The high-tenure, two-year-contract segment (Cluster TBD, TBD% of customers, churn rate TBD%) represents the company's most stable revenue base. Retention spend here has low ROI — these customers are not at risk. Invest instead in migrating month-to-month customers from the high-churn segment into this profile via contract upgrade incentives (price locks, bundled add-ons, loyalty discounts).

**2. Intervene early on new high-spend customers.**
The new customer, high monthly spend segment (Cluster TBD, TBD% of customers, churn rate TBD%) churns at above-average rates despite paying premium prices. This is a value-perception problem: these customers are spending more than average but haven't yet formed loyalty. The intervention window is the first 90 days. Proactive onboarding, usage check-ins, and TechSupport enrollment have been shown to reduce early churn in similar telco datasets.

**3. Target contract upgrades at the medium-tenure month-to-month segment.**
Month-to-month customers who have stayed for 12–24 months (Cluster TBD) are the highest-ROI conversion targets. They have already demonstrated some loyalty but retain the flexibility to leave at any point. A personalised offer at the 12-month mark — price-locked annual contract with one add-on included — can shift a meaningful portion of this segment into the two-year tier.

**4. Deprioritise retention spend on the low-tenure, low-spend, no-internet segment.**
The segment with minimal service engagement (phone-only, no internet, low monthly charges) has a churn profile that reflects end-of-contract behaviour rather than dissatisfaction. Retention spend here is wasteful — these customers are churning because their need is fulfilled, not because they are unhappy. A better use of budget is win-back: service upgrade offers sent 30 days before their expected churn date targeting service upsell rather than retention.

**5. Use segment membership as a feature in the churn prediction model.**
The natural next step is to pass each customer's segment label as a feature into the telco-churn-predictor model, or to train per-segment churn models and compare their AUC against the global baseline. Within-segment populations are more homogeneous, which typically yields 3–8 AUC points of improvement on the highest-churn segment.

## Connecting Segmentation to Churn Prediction

The natural next step after segmentation is churn prediction within each segment — a model trained specifically on high-risk segment customers will outperform a global model because the within-segment population is more homogeneous. In telco-churn-predictor I built the global model. The extension is to train per-segment models and compare their ROC-AUC against the global baseline. This is a standard MLOps pattern for improving model performance on heterogeneous populations.

Concretely: fit a Random Forest on the highest-churn segment only, tune threshold on its own precision-recall curve, and compare that model's AUC against the global model evaluated on the same subset. On most real-world datasets the segment-specific model wins by 3–8 AUC points because it is not distorted by the very different churn dynamics in other segments.

## Azure App Service Deployment

This is the third Flask app I have deployed to Azure App Service, after telco-churn-predictor and spam-classifier. The deployment process is now well-established:

- Create App Service Plan at B1 tier via CLI
- Scale down to F1 free tier via portal (F1 cannot be provisioned via CLI on this account — quota restriction)
- Set startup command: `gunicorn --bind=0.0.0.0:8000 --timeout 600 app:app`
- Set `SCM_DO_BUILD_DURING_DEPLOYMENT=true`
- Deploy via: `az webapp deployment source config-zip`

One new consideration for this project: the `/api/cluster-data` endpoint returns 2,000 rows of PCA coordinates on page load. On the F1 free tier (shared CPU, 1 GB RAM) this is within limits, but response time is ~800ms. For production this endpoint would be pre-computed and cached.

```bash
az group create --name customer-segmentation-rg --location westeurope

az appservice plan create \
  --name customer-segmentation-plan \
  --resource-group customer-segmentation-rg \
  --sku B1 --is-linux

# Scale to F1 via Azure portal after provisioning

az webapp create \
  --name customer-segmentation \
  --resource-group customer-segmentation-rg \
  --plan customer-segmentation-plan \
  --runtime "PYTHON:3.11"

az webapp config set \
  --name customer-segmentation \
  --resource-group customer-segmentation-rg \
  --startup-file "gunicorn --bind=0.0.0.0:8000 --timeout 600 app:app"

az webapp config appsettings set \
  --name customer-segmentation \
  --resource-group customer-segmentation-rg \
  --settings SCM_DO_BUILD_DURING_DEPLOYMENT=true

cd customer-segmentation
zip -r deploy.zip . -x "*.git*" -x "venv/*" -x "__pycache__/*" -x "*.ipynb_checkpoints*"

az webapp deployment source config-zip \
  --name customer-segmentation \
  --resource-group customer-segmentation-rg \
  --src deploy.zip
```